In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Plot Continuous Blood Pressures from two subjects
from matplotlib import colors
from scipy.signal import find_peaks
from pathlib import Path


# Blood Pressure and Flow in the Human Vasculature

## Exercise: Blood Flow Through an Artery — Poiseuille's Law

An artery has a length of 100 mm. The blood viscosity is 3.5 mPa·s, and the pressure difference across the artery is 1 mmHg (133.3 Pa). The artery diameter varies from 0.1 mm to 30 mm.

Use Poiseuille's law:

$$
Q = \frac{\pi a^4}{8\eta L}\Delta P
$$

where:

* $Q$ = volumetric flow rate [m³/s]
* $a$ = artery radius [m]
* $\eta$ = blood viscosity [Pa·s]
* $L$ = artery length [m]
* $\Delta P$ = pressure difference [Pa]

The given values are:

$$
L = 100\text{ mm} = 0.100\text{ m}
$$

$$
\eta = 3.5\text{ mPa·s} = 3.5\cdot 10^{-3}\text{ Pa·s}
$$

$$
\Delta P = 1\text{ mmHg} \approx 133.3\text{ Pa}
$$

### 1. Define the function

Write a Python function `poiseuille_flow` that calculates the blood flow rate $Q$. 

### 2. Calculate and plot

Use the function to calculate and plot the blood flow rate $Q$ against artery diameter $d$ for:

$$
0.1\text{ mm} \leq d \leq 30\text{ mm}
$$

Plot the diameter in mm on the x-axis and the flow rate in m³/s on the y-axis. Briefly describe the relationship between artery diameter and blood flow.


### Answer
#### 1

In [ ]:
# Poiseuille's Law: Q = (π * ΔP * r^4) / (8 * η * L)
def poiseuille_flow(dP, radius, viscosity, length):
    """dP [Pa], radius [m], viscosity [Pa·s], length [m] -> Q [m³/s]"""
    return (np.pi * dP * radius**4) / (8 * viscosity * length)


#### 2

In [ ]:

diameter = np.linspace(0.1e-3, 30e-3, 300)  # Diameter from 0.1 mm to 30 mm [m]
dP = 133.3    # Pa
viscosity = 3.5e-3  # Pa.s
length = 0.1  # m
# Calculate flow
Q = poiseuille_flow(dP, diameter / 2, viscosity, length)

plt.figure(figsize=(8,5))
# Plot
plt.plot(diameter * 1e3, Q)
plt.xlabel('Artery diameter (mm)')
plt.ylabel('Flow rate Q (m³/s)')
plt.title("Poiseuille's Law: Flow vs Vessel Diameter")
plt.grid(True)
plt.show()


Flowet vokser med radius i 4. potens ($Q \propto r^4 \propto d^4$). En fordobling af diameteren giver 16 gange så stort flow ved samme trykforskel, og en halvering giver 1/16. Derfor er små kar (arterioler) dem med størst modstand, og små ændringer i deres diameter (vasokonstriktion/-dilatation) styrer fordelingen af blodflowet.

## Exercise: Pressure 1

Denne opgave omhandler `Section 5.9`. 

Forestil dig, at du har en mand, som ligger fladt, så tyngdekraftens rolle for den energi der benyttes i kroppen er minimal ($h=0$). 
Lad manden have en stroke volume på $71mL$, og en puls på 70 slag per minut, Antag et konstant blodtryk på 130mmHg, densitet af blod er $\rho=1.05 g/cm^{3}$, og arealet af Aorta er $3.5cm^2$.

1. Beregn gennemsnitlige minutvolumen?
2. Beregn end pressure blodtryk i aorta?
    - Hints:
        - Når man ligger ned påvirker tyngdekraften ikke trykket
        - Det betyder at vi skal regne det givne tryk ud, men tilføje trykket mht hastigheden
        - Hastigheden af væske i et rør, er givet ved $v=V/A$, hvor $V$ er volumen og $A$ er tværsnits arealet af røret.


In [ ]:
densitity_blood = 1.05 # g/mL = g/cm^3
HR = 70 # slag/min
SV = 71 # mL/slag
A_aorta = 3.5 # cm^2
# Exercise:
P = 130  # mmHg

# Beregn gennemsnitlig minut volumen
CO = HR * SV

# Beregn end pressure blodtryk?
vol_per_second = CO / 60  # mL/s
v = vol_per_second / A_aorta  # cm/s

# Kinetisk tryk 1/2*rho*v^2 i SI-enheder (Pa), omregnet til mmHg
rho_SI = densitity_blood * 1000  # kg/m^3
v_SI = v / 100  # m/s
P_kinetic = 0.5 * rho_SI * v_SI**2 / 133.322  # mmHg
P_end_pressure = P + P_kinetic


print(f"Cardiac Output (CO): {CO} mL/min")
print(f"Blood velocity in aorta: {v:.1f} cm/s")
print(f"Kinetic pressure: {P_kinetic:.2f} mmHg")
print(f"End Pressure Blood Pressure: {P_end_pressure:.2f} mmHg")

# Pressure 2

`files/autonomic-aging-cardiovascular/0001.csv` indeholder en del data, som er optaget med $fs=100Hz$. 

### Exercise: Know your data!

1. Åben en af filerne i excel og derefter i vs code for at se hvordan dataen er gemt. Til denne opgave skal vi holde fokus på tid og blodtryk kolonnerne. Hvad hedder de kolonner?

`time_s, ECG1, ECG2, NIBP`


In [ ]:
df0001 = pd.read_csv(Path("files/autonomic-aging-cardiovascular/0001.csv"), delimiter=",")
fs = 100 # samplingrate
# Show data df0001
df0001

# Arterielle blodtryk

In [ ]:

bp0001 = df0001["NIBP"].to_numpy()
time = df0001["time_s"].to_numpy()
peaks_indecies_top, _ = find_peaks(bp0001,prominence=2*bp0001.std(), distance=fs*0.3) # du kan finde highpoints, ved bare at bruge det positive signalet
peaks_indecies_bot, _ = find_peaks(-bp0001, prominence=2*bp0001.std(),  distance=fs*0.3) # du kan finde lowpoints, ved bare at bruge det negative signalet



# Plot Continuous Blood Pressures with peak pressures 
plt.plot(time, bp0001, label="0001")
plt.scatter(time[peaks_indecies_bot], bp0001[peaks_indecies_bot])
plt.scatter(time[peaks_indecies_top], bp0001[peaks_indecies_top], marker="x")

plt.ylabel("BP [mmHg]")
plt.xlabel("time [s]")
plt.legend(loc='upper right')
plt.xlim((0,12))
plt.title(f"Continuous Blood Pressure Over 10 seconds, samplerate={fs}Hz")
plt.show()

## Exercise: Arterial Bloodpressure Estimates
I plottet `Continuous Blood Pressure Over 10 seconds, samplerate=100Hz` ovenover kan du se 10 sekunders optagelse af blodtryk fra personen.
1. Beregn og plot det gennemsnitlige arterielle blodtryk for hvert hjerteslag via `Formel 5.9.8` for personen. *Hint: se `Figure 5.9.3` i bogen.*:
    - Hint: hvis du skal bruge pulsen fra hvert hjerteslag, kan du bruge `peaks_indecies_bot` værdierne. 
$$ P_A = \frac{\int_{t_1}^{t_2} P(t) dt}{t_2-t_1} $$

In [ ]:

MAP = []
MAP_time = []

for p1,p2,t in zip( peaks_indecies_top[0:], peaks_indecies_top[1:], time[peaks_indecies_top[0:]]):
    # Loop der finder hvert hjerteslag, og gemmer i signal
    signal = bp0001[p1:p2]


2. Beregne og plot det gennemsnitlige arterielle blodtryk ved hjælp af den simplere model: `Formel 5.9.9`. 
    $$
    MAP = P_{dia}+\frac{P_{sys}-P_{dia}}{3}
    $$

In [ ]:
MAP_est = []
MAP_est_tid = []
# Hvert slag: diastolisk minimum efterfulgt af det næste systoliske maksimum
for l,p,t in zip( bp0001[peaks_indecies_bot], bp0001[peaks_indecies_top[1:]], time[peaks_indecies_bot]):
    # Add results to estimate array 
    # MAP_est_tid.append(t)
    # bp = ???
    # MAP_est.append(bp)
    ...

3. Sammenlign resultaterne for `Formel 5.9.8` og `Formel 5.9.9`, og med egne ord argumenter for både fordele og ulemper ved hver metode.
   

#### Answer
